In [1]:
import matplotlib.pyplot as plt
from nn import NN
import os
import numpy as np
import random
import cv2 

In [2]:
PATH = "trainingSet"
EPOCHS = 10
EVAL_EVERY = 1
labels = sorted(os.listdir(PATH), key=int)
w, h = 28, 28
lr = 5e-3
nn = NN(w * h, 16, 10, lr)

In [3]:
def getY( x ):
    return np.array([ int(x==i) for i in range(10) ]).reshape(-1,1)


In [4]:
data = []

for label in labels:
    path = os.path.join( PATH, label )
    img_files = os.listdir( path )
    for img_name in img_files:
        file_path = os.path.join( path, img_name )
        data.append({
            "file" : file_path,
            "label" : label,
            "Y" : getY(int(label))
        })

random.shuffle( data )

In [5]:
def load_input(file_path):
    ip = cv2.imread(file_path)
    ip = cv2.cvtColor(ip, cv2.COLOR_BGR2GRAY)
    return ip.reshape(w * h, 1) / 255.0


def evaluate_dataset(model, dataset):
    y_true = []
    y_pred = []
    losses = []
    eps = 1e-12

    for img_data in dataset:
        ip = load_input(img_data["file"])
        probs, pred_idx = model.predict(ip)
        target_idx = int(np.argmax(img_data["Y"]))

        y_true.append(target_idx)
        y_pred.append(int(pred_idx))

        p_true = float(probs[target_idx, 0])
        losses.append(-np.log(max(p_true, eps)))

    return np.array(y_true), np.array(y_pred), float(np.mean(losses))


def classification_metrics(y_true, y_pred, num_classes=10):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1

    total = cm.sum()
    correct = np.trace(cm)
    accuracy = correct / total if total else 0.0

    tp = np.diag(cm).astype(float)
    predicted = cm.sum(axis=0).astype(float)
    actual = cm.sum(axis=1).astype(float)

    precision_per_class = np.divide(
        tp, predicted, out=np.zeros_like(tp), where=predicted != 0
    )
    recall_per_class = np.divide(
        tp, actual, out=np.zeros_like(tp), where=actual != 0
    )
    f1_per_class = np.divide(
        2 * precision_per_class * recall_per_class,
        precision_per_class + recall_per_class,
        out=np.zeros_like(tp),
        where=(precision_per_class + recall_per_class) != 0,
    )

    return {
        "confusion_matrix": cm,
        "accuracy": float(accuracy),
        "precision": float(np.mean(precision_per_class)),
        "recall": float(np.mean(recall_per_class)),
        "f1": float(np.mean(f1_per_class)),
    }

In [ ]:
for e in range(EPOCHS):
    try:
        random.shuffle(data)
        for img_data in data:
            ip = load_input(img_data["file"])
            nn.train(ip, img_data["Y"], m=1)

        should_eval = ((e + 1) % EVAL_EVERY == 0) or (e == 0) or (e + 1 == EPOCHS)
        if should_eval:
            y_true, y_pred, epoch_loss = evaluate_dataset(nn, data)
            metrics = classification_metrics(y_true, y_pred, num_classes=10)
            print(
                f"Epoch {e + 1}/{EPOCHS} | "
                f"loss: {epoch_loss:.4f} | "
                f"acc: {metrics['accuracy']:.4f} | "
                f"precision: {metrics['precision']:.4f} | "
                f"recall: {metrics['recall']:.4f} | "
                f"f1: {metrics['f1']:.4f}"
            )
    except KeyboardInterrupt:
        print("\nTraining interrupted. Evaluating final model...")
        

In [ ]:
y_true, y_pred, final_loss = evaluate_dataset(nn, data)
final_metrics = classification_metrics(y_true, y_pred, num_classes=10)

print("Final Training Metrics")
print(f"loss: {final_loss:.4f}")
print(f"accuracy: {final_metrics['accuracy']:.4f}")
print(f"precision: {final_metrics['precision']:.4f}")
print(f"recall: {final_metrics['recall']:.4f}")
print(f"f1: {final_metrics['f1']:.4f}")
print("Hyperparameters:")
print(f"Learning Rate: {nn.learning_rate}")
print(f"Epochs: {EPOCHS}")

plt.imshow(final_metrics["confusion_matrix"], cmap='viridis')

nn.save("basic.nn")

Final Training Metrics
loss: 0.1546
accuracy: 0.9535
precision: 0.9539
recall: 0.9534
f1: 0.9531
Hyperparameters:
Learning Rate: 0.005


NameError: name 'epochs' is not defined